# 🦷 DENTEX 4-Class Abnormal Tooth Detection Benchmark (Google Colab 1-Click)

**GitHub Repository:** [https://github.com/tmanhococ/dentex-benchmark](https://github.com/tmanhococ/dentex-benchmark)  
**Authors:** Dental Pathology AI Research Group  
**Target Runtime:** Google Colab (GPU T4 / V100 / A100)

---

## 📖 Phần I: Tổng Quan Bài Toán & Phân Loại Bệnh Lý Lâm Sàng

Bài toán tập trung vào phát hiện và phân loại **4 tổn thương / bất thường răng miệng phổ biến nhất** trên phim chụp X-quang răng toàn cảnh (panoramic dental radiograph) từ bộ dữ liệu **DENTEX Challenge** (MICCAI 2023):

1. **Caries (ID: 1):** Sâu răng giai đoạn sớm / tổn thương men răng và ngà răng nông.
2. **Deep Caries (ID: 2):** Sâu răng mức độ nặng, tổn thương sâu tiến sát hoặc đã thâm nhiễm vào buồng tủy (pulp chamber).
3. **Periapical Lesion (ID: 3):** Tổn thương thấu quang quanh chóp / u nang viêm chân răng (radiolucency / cyst tại root apex).
4. **Impacted Tooth (ID: 4):** Răng ngầm / răng khôn kẹt cơ học không thể mọc bình thường lên cung hàm.

*Lưu ý y khoa:* Răng khỏe mạnh bình thường đóng vai trò là hậu cảnh (background) để mô hình tập trung phân biệt và định vị chính xác bệnh lý lâm sàng.


## 🔬 Phần II: Cơ Sở Lý Thuyết & Công Thức Toán Học Của Mô Hình

Notebook này triển khai và đối sánh 2 kiến trúc tiên tiến đại diện cho 2 trường phái giải thuật:

### 1. HierarchicalNet (Multi-Scale Feature Hierarchy)
Do các tổn thương nha khoa có sự chênh lệch kích thước rất lớn ($Caries$ có thể chỉ chiếm vài chục pixel trong khi $Impacted\ Tooth$ hay nang chân răng lớn chiếm hàng trăm pixel), **HierarchicalNet** sử dụng Feature Pyramid Network (FPN) trên nền xương sống ResNet-50.

#### a. Trích xuất đặc trưng Bottom-Up (ResNet-50 Backbone)
Ảnh đầu vào $I \in \mathbb{R}^{3 \times H \times W}$ đi qua 4 tầng dư với độ thu nhỏ lũy tiến:
$$C_2, C_3, C_4, C_5 = \text{ResNet50}(I)$$
trong đó số kênh tương ứng là $256, 512, 1024, 2048$ và stride lần lượt là $4, 8, 16, 32$.

#### b. Kết nối ngang & Đường truyền Top-Down (FPN)
Các đặc trưng ngữ nghĩa mức cao được truyền ngược xuống và hợp nhất với đặc trưng không gian chi tiết mức thấp:
$$P_5 = \text{Conv}_{3\times3}(\text{Conv}_{1\times1}(C_5))$$
$$P_i = \text{Conv}_{3\times3}\left(\text{Conv}_{1\times1}(C_i) + \text{Upsample}_{2\times}(P_{i+1})\right), \quad i \in \{4, 3, 2\}$$
Tất cả các tầng kim tự tháp $\{P_2, P_3, P_4, P_5\}$ đều có cùng số kênh chuẩn hóa $d = 256$.

#### c. Đầu dự đoán phân cấp & Hàm mất mát (Loss Function)
Đầu dự đoán phân cấp tổng hợp ngữ cảnh đa tỉ lệ và ánh xạ ra xác suất lớp $\hat{p} \in \mathbb{R}^{K+1}$ ($K=4$ bệnh lý + 1 background) cùng tọa độ box chuẩn hóa $\hat{b} \in [0, 1]^4$:
$$\mathcal{L}_{HierarchicalNet} = \mathcal{L}_{CE}(\hat{p}, p^*) + \lambda_{reg} \cdot \mathbf{1}_{[p^* \ge 1]} \mathcal{L}_{L1}(\hat{b}, b^*)$$

---

### 2. DiffusionNet (Progressive Denoising Diffusion Detector)
**DiffusionNet** tiếp cận bài toán phát hiện vật thể dưới dạng tiến trình khuếch tán xác suất ngược (Reverse Denoising Diffusion Process), biến đổi các bounding box ngẫu nhiên từ phân phối chuẩn thành các box chuẩn xác bao quanh bệnh lý.

#### a. Tiến trình khuếch tán thuận (Forward Noising Process)
Cho bounding box gốc $z_0 \in [0, 1]^4$. Tại bước thời gian liên tục $t \in [0, T]$ ($T=1000$):
$$q(z_t | z_0) = \mathcal{N}\left(z_t; \sqrt{\bar{\alpha}_t} z_0, (1 - \bar{\alpha}_t) \mathbf{I}\right)$$
với $\alpha_t = 1 - \beta_t$ và $\bar{\alpha}_t = \prod_{s=1}^t \alpha_s$.

#### b. Nhúng bước thời gian Sinusoidal & Điều kiện hóa
Bước thời gian $t$ được mã hóa qua hàm điều hòa:
$$e_k(t) = \begin{cases} \sin\left(t / 10000^{2k/d}\right), & k \text{ chẵn} \\ \cos\left(t / 10000^{2k/d}\right), & k \text{ lẻ} \end{cases}$$
Biểu diễn điều kiện hóa kết hợp đặc trưng toàn cục $F = \text{Backbone}(I)$ và vector thời gian:
$$Cond = \text{GlobalPool}(F) + \text{MLP}(e(t))$$

#### c. Khử nhiễu tiến trình nghịch đảo & Hàm mất mát
Mạng `DynamicDiffusionHead` ước lượng box sạch $\hat{z}_0$ và phân loại tổn thương:
$$\hat{z}_0, \hat{c} = f_\theta(z_t, Cond)$$
$$\mathcal{L}_{DiffusionNet} = \mathcal{L}_{cls}(\hat{c}, c^*) + \lambda_{L1} \|\hat{z}_0 - z_0\|_1 + \lambda_{giou} \mathcal{L}_{giou}(\hat{z}_0, z_0)$$


## ⚙️ Phần III: Quy Trình Huấn Luyện & Cơ Chế Chống Tràn VRAM / NaN

Để giải quyết triệt để lỗi **CUDA Out of Memory (OOM)** trên Colab T4 (15 GB) và lỗi **Loss: NaN** khi fine-tune, hệ thống áp dụng các tiêu chuẩn ổn định công nghiệp kế thừa từ template `cv-dinoresnet.ipynb`:

```
[ Ảnh Panorama Gốc ~2800x1300 px ]
               │
               ▼
[ Multi-Scale Resize: Short Edge 480..800 px, Max Edge 1333 px ]
               │
               ▼
[ DataLoader: batch_size = 1, num_workers = 2 ]
               │
               ▼
[ AMP Autocast: torch.amp.autocast('cuda') (FP16/BF16) ]
               │
               ▼
[ Forward Pass: HierarchicalNet / DiffusionNet ]
               │
               ▼
[ Loss Calculation & Chia Cho Gradient Accumulation (16) ]
               │
               ▼
[ Scaled Backward: scaler.scale(loss).backward() ]
               │
               ▼
     [ Step % 16 == 0 ? ]
     ├── YES ──► [ scaler.unscale_(optimizer) ]
     │                │
     │                ▼
     │           [ Kiểm Tra Gradients Hữu Hạn ]
     │           ├── Inf/NaN ──► [ Bỏ qua batch lỗi, giảm scale, tránh NaN model ]
     │           └── Finite  ──► [ clip_grad_norm_(0.1) ──► optimizer.step() ]
     └── NO  ──► Tiếp tục tích lũy gradient
               │
               ▼
[ Cuối Mỗi Epoch (Max 30 Epochs) ]
               │
               ├── val_loss tốt nhất ──► Lưu outputs/<model>/best.pth
               └── Luôn cập nhật      ──► Lưu outputs/<model>/last.pth
```

- **Bộ nhớ tiêu thụ:** Nhờ `batch_size = 1` kết hợp tích lũy gradient 16 bước, VRAM thực tế luôn duy trì $< 6$ GB, chạy mượt mà trên GPU T4 mà không bao giờ gặp OOM.
- **Ổn định số học:** Gradient clipping $0.1$ và cơ chế unscale guard loại bỏ hoàn toàn hiện tượng bùng nổ gradient gây NaN.
- **Giới hạn số Epoch:** Huấn luyện tối đa **30 epochs**, tự động ghi nhận lại checkpoint tối ưu nhất (`best.pth`) và checkpoint cuối cùng (`last.pth`).


## 📊 Phần IV: Lý Thuyết Các Chỉ Số Đánh Giá (Evaluation Metrics)

### 1. Intersection over Union (IoU)
$$\text{IoU}(B_p, B_{gt}) = \frac{\text{Area}(B_p \cap B_{gt})}{\text{Area}(B_p \cup B_{gt})}$$

### 2. Average Precision (AP) & COCO Metrics
Đường cong Precision-Recall $P(r)$ cho từng lớp bệnh lý:
$$\text{AP}_c = \int_0^1 P_c(r) dr \approx \sum_{k=1}^N (R_k - R_{k-1}) \max_{\tilde{r} \ge R_k} P(\tilde{r})$$
- **$mAP_{50}$:** Trung bình AP của 4 bệnh lý tại ngưỡng $\text{IoU} = 0.50$.
- **$mAP_{75}$:** Ngưỡng đánh giá khắt khe tại $\text{IoU} = 0.75$.
- **$mAP_{50:95}$:** Chuẩn mực quốc tế COCO, tính trung bình qua 10 ngưỡng IoU từ $0.50$ đến $0.95$ (bước nhảy $0.05$):
  $$mAP_{50:95} = \frac{1}{10} \sum_{\tau \in \{0.50, 0.55, \dots, 0.95\}} mAP_\tau$$
- **Average Recall (AR@100):** Tỉ lệ phát hiện tổn thương tối đa khi cho phép mô hình dự đoán tối đa 100 boxes mỗi ảnh.

### 3. Chỉ Số Phân Loại Lâm Sàng (Clinical F1-Score)
Khớp dự đoán với Ground Truth ở ngưỡng $\text{IoU} \ge 0.50$ để xác định $TP, FP, FN$:
$$\text{Precision}_c = \frac{TP_c}{TP_c + FP_c}, \quad \text{Recall}_c = \frac{TP_c}{TP_c + FN_c}$$
$$\text{F1}_{macro} = \frac{1}{4} \sum_{c=1}^4 \frac{2 \cdot \text{Precision}_c \cdot \text{Recall}_c}{\text{Precision}_c + \text{Recall}_c}$$


## 🛠️ Cell 0: Cấu Hình Thực Thi


In [ ]:
REPO_URL = 'https://github.com/tmanhococ/dentex-benchmark.git'
BRANCH = 'main'

USE_DRIVE = True     # Tự động mount Google Drive để lưu checkpoints lâu dài
DRIVE_DIR = '/content/drive/MyDrive/dentex-benchmark'
REQUIRE_GPU = True   # Yêu cầu bắt buộc GPU (Colab T4)
SKIP_TRAINED = True  # Bỏ qua mô hình nếu đã huấn luyện xong 30 epoch
SEED = 42

# 2 Mô hình mục tiêu: Huấn luyện tối đa 30 epochs
MODELS = {
    'hierarchical_net': {'epochs': 30, 'lr': 0.0001},
    'diffusion_net':    {'epochs': 30, 'lr': 0.0001},
}


## 💻 Cell 1: Kiểm Tra GPU & Mount Google Drive


In [ ]:
from pathlib import Path
import torch

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'✅ GPU Sẵn Sàng: {props.name} ({props.total_memory / 1024 ** 3:.2f} GB VRAM)')
elif REQUIRE_GPU:
    raise RuntimeError('❌ Không tìm thấy GPU. Hãy vào Runtime → Change runtime type → Chọn T4 GPU rồi chạy lại.')
else:
    print('⚠️ Chạy trên CPU — tốc độ sẽ rất chậm.')

DRIVE_OK = False
if USE_DRIVE and IN_COLAB:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        Path(DRIVE_DIR).mkdir(parents=True, exist_ok=True)
        DRIVE_OK = True
        print(f'✅ Google Drive Đã Kết Nối: {DRIVE_DIR}')
    except Exception as e:
        print(f'⚠️ Không thể kết nối Google Drive ({e}). Kết quả sẽ lưu tạm tại Colab local.')


## 📥 Cell 2: Tải Mã Nguồn Mới Nhất Từ GitHub


In [ ]:
import os, sys, codecs, subprocess
from pathlib import Path

WORK_DIR = Path('/content/dentex-benchmark')

def run(cmd, capture=False):
    print(f'$ {cmd}', flush=True)
    env = {**os.environ, 'PYTHONUNBUFFERED': '1'}
    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, env=env)
    decoder = codecs.getincrementaldecoder('utf-8')(errors='replace')
    chunks = []
    for raw in iter(lambda: proc.stdout.read1(4096), b''):
        text = decoder.decode(raw)
        sys.stdout.write(text)
        sys.stdout.flush()
        if capture:
            chunks.append(text)
    if proc.wait() != 0:
        raise RuntimeError(f'Lệnh thất bại (exit code {proc.returncode}): {cmd}')
    return ''.join(chunks)

if (WORK_DIR / '.git').exists():
    print('🔄 Cập nhật repository hiện có...')
    run(f'git -C {WORK_DIR} fetch origin {BRANCH} && git -C {WORK_DIR} reset --hard origin/{BRANCH}')
else:
    print('🚀 Cloning repository từ GitHub...')
    run(f'git clone -b {BRANCH} {REPO_URL} {WORK_DIR}')

os.chdir(WORK_DIR)
if str(WORK_DIR) not in sys.path:
    sys.path.insert(0, str(WORK_DIR))
print(f'✅ Thư mục làm việc: {Path.cwd()}')


## 📦 Cell 3: Kiểm Tra & Cài Đặt Thư Viện


In [ ]:
from importlib import metadata
from packaging.requirements import Requirement

ALTERNATIVES = {
    'opencv-python': ['opencv-python-headless', 'opencv-contrib-python', 'opencv-contrib-python-headless'],
}

def installed_version(name):
    for dist in [name] + ALTERNATIVES.get(name, []):
        try:
            return metadata.version(dist)
        except metadata.PackageNotFoundError:
            continue
    return None

to_install = []
for line in Path('requirements.txt').read_text(encoding='utf-8').splitlines():
    line = line.strip()
    if not line or line.startswith('#'):
        continue
    req = Requirement(line)
    ver = installed_version(req.name)
    if ver is None or (req.specifier and not req.specifier.contains(ver, prereleases=True)):
        to_install.append(line)

if to_install:
    print(f'📦 Cài đặt thư viện: {to_install}')
    run(f'{sys.executable} -m pip install -q ' + ' '.join(f'"{p}"' for p in to_install))
else:
    print('✅ Tất cả thư viện phụ thuộc đã sẵn sàng.')

run(f'{sys.executable} scripts/setup_env.py')


## 🗂️ Cell 4: Chuẩn Bị Dữ Liệu DENTEX (Auto Download & COCO 4-Class)


In [ ]:
import json

train_ann = Path('data/processed/instances_train.json')
val_ann = Path('data/processed/instances_val.json')

drive_processed = Path(DRIVE_DIR) / 'data/processed' if DRIVE_OK else None
if drive_processed and (drive_processed / 'instances_train.json').exists():
    print(f'📦 Khôi phục dữ liệu đã xử lý từ Google Drive ({drive_processed})...')
    run(f'mkdir -p data && cp -ru "{Path(DRIVE_DIR)}/data" .')

if not (train_ann.exists() and val_ann.exists()):
    print('🌐 Đang tải và chuẩn bị bộ dữ liệu DENTEX từ Hugging Face Hub...')
    run(f'{sys.executable} scripts/download_and_prepare.py --data_dir data --seed {SEED}')
    if DRIVE_OK:
        print('💾 Đồng bộ dữ liệu lên Google Drive...')
        run(f'mkdir -p "{Path(DRIVE_DIR)}/data" && cp -ru data/* "{Path(DRIVE_DIR)}/data/"')

with open(train_ann, encoding='utf-8') as f:
    t_data = json.load(f)
with open(val_ann, encoding='utf-8') as f:
    v_data = json.load(f)

print(f'✅ Dữ liệu sẵn sàng:')
print(f'  - Train: {len(t_data["images"])} ảnh, {len(t_data["annotations"])} nhãn tổn thương.')
print(f'  - Validation: {len(v_data["images"])} ảnh, {len(v_data["annotations"])} nhãn tổn thương.')


## 🏋️ Cell 5: Huấn Luyện HierarchicalNet & DiffusionNet (30 Epochs, Zero OOM)


In [ ]:
import shutil

NAME_MAP = {
    'hierarchical_net': 'HierarchicalNet (Multi-Scale Feature Hierarchy)',
    'diffusion_net':    'DiffusionNet (Progressive Box Denoising)'
}

for i, (m_key, m_cfg) in enumerate(MODELS.items(), 1):
    epochs = m_cfg['epochs']
    lr = m_cfg['lr']
    label = NAME_MAP.get(m_key, m_key)
    out_dir = Path('outputs') / m_key
    drive_out = Path(DRIVE_DIR) / 'outputs' / m_key if DRIVE_OK else None

    # Khôi phục từ Drive nếu đã có
    if drive_out and (drive_out / 'best.pth').exists() and not (out_dir / 'best.pth').exists():
        print(f'📦 Khôi phục checkpoint {m_key} từ Google Drive...')
        out_dir.mkdir(parents=True, exist_ok=True)
        for ckpt in drive_out.glob('*.pth'):
            shutil.copy2(ckpt, out_dir / ckpt.name)

    if SKIP_TRAINED and (out_dir / 'best.pth').exists() and (out_dir / 'last.pth').exists():
        print(f'⏭️ [{i}/{len(MODELS)}] {label}: Đã có checkpoint hoàn chỉnh (best.pth, last.pth). Bỏ qua huấn luyện.')
        continue

    print(f'\n============================================================')
    print(f'🏋️ [{i}/{len(MODELS)}] Huấn Luyện {label}: {epochs} epochs, lr={lr}')
    print(f'============================================================')

    try:
        run(f'{sys.executable} scripts/train.py --model {m_key} --epochs {epochs} --lr {lr}')
    except Exception as e:
        print(f'❌ Lỗi khi huấn luyện {m_key}: {e}')
        continue

    if DRIVE_OK:
        print(f'💾 Đang sao lưu checkpoint {m_key} lên Google Drive...')
        drive_out.mkdir(parents=True, exist_ok=True)
        for f in out_dir.glob('*'):
            shutil.copy2(f, drive_out / f.name)
        print('✅ Đã sao lưu thành công lên Drive.')


## 📈 Cell 6: Đánh Giá Độc Lập Trên Tập Validation & Xuất Báo Cáo


In [ ]:
print('=== Đánh Giá Đối Soát Toàn Diện Trên Tập Validation ===')
run(f'{sys.executable} scripts/evaluate.py --compare_all')

if DRIVE_OK:
    for f in Path('outputs').glob('*.*'):
        shutil.copy2(f, Path(DRIVE_DIR) / 'outputs' / f.name)
    print('✅ Đã sao lưu báo cáo khoa học sang Drive.')


## 📊 Cell 7: Hiển Thị Đồ Thị Huấn Luyện & Biểu Đồ So Sánh


In [ ]:
from IPython.display import display, Image as DisplayImage, Markdown

for m_key in MODELS:
    curve_file = Path('outputs') / m_key / 'training_curves.png'
    if curve_file.exists():
        print(f'📈 Đồ thị huấn luyện Loss: {m_key}')
        display(DisplayImage(str(curve_file)))

comp_plot = Path('outputs/metrics_comparison.png')
if comp_plot.exists():
    print('📊 Biểu đồ so sánh hiệu năng tổng thể:')
    display(DisplayImage(str(comp_plot)))


## 📄 Cell 8: Hiển Thị Bảng Số Liệu Markdown & Mã Nguồn LaTeX


In [ ]:
summary_file = Path('outputs/benchmark_summary.md')
if summary_file.exists():
    print('📋 Bảng tóm tắt kết quả (Markdown):')
    display(Markdown(summary_file.read_text(encoding='utf-8')))

tex_file = Path('outputs/benchmark_table.tex')
if tex_file.exists():
    print('\n==========================================================')
    print('📄 MÃ NGUỒN BẢNG LATEX (Dùng cho Overleaf / Báo cáo khoa học):')
    print('==========================================================\n')
    print(tex_file.read_text(encoding='utf-8'))


## 🖼️ Cell 9: Trực Quan Hóa Bounding Box Dự Đoán Trên Ảnh X-Quang Thực Tế


In [ ]:
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from dentex_bench.config import BenchmarkConfig
from dentex_bench.models import HierarchicalNetRunner, DiffusionNetRunner

CLASS_COLORS = {
    1: ('#E69F00', 'Caries'),
    2: ('#D55E00', 'Deep Caries'),
    3: ('#0072B2', 'Periapical Lesion'),
    4: ('#009E73', 'Impacted Tooth')
}

val_imgs = list(Path('data/raw').glob('**/*.png')) + list(Path('data/raw').glob('**/*.jpg'))
if val_imgs:
    sample_img_path = val_imgs[0]
    print(f'🖼️ Hiển thị dự đoán trên ảnh mẫu: {sample_img_path.name}')

    bgr = cv2.imread(str(sample_img_path))
    if bgr is not None:
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        tensor = torch.from_numpy(rgb).permute(2, 0, 1).float() / 255.0

        cfg = BenchmarkConfig.from_auto_env()
        fig, axes = plt.subplots(1, 2, figsize=(16, 7))

        runner_types = [
            ('hierarchical_net', 'HierarchicalNet', HierarchicalNetRunner),
            ('diffusion_net', 'DiffusionNet', DiffusionNetRunner)
        ]

        for ax, (m_key, title, runner_cls) in zip(axes, runner_types):
            ax.imshow(rgb)
            ax.set_title(f'{title} Predictions (Score >= 0.3)', fontsize=13)
            ax.axis('off')

            ckpt = Path('outputs') / m_key / 'best.pth'
            if ckpt.exists():
                runner = runner_cls(cfg)
                runner.build_model()
                runner.load_checkpoint(ckpt)
                preds = runner.predict([tensor])[0]

                keep = preds['scores'] >= 0.3
                boxes = preds['boxes'][keep].cpu().numpy()
                labels = preds['labels'][keep].cpu().numpy()
                scores = preds['scores'][keep].cpu().numpy()

                for b, l, s in zip(boxes, labels, scores):
                    color, cname = CLASS_COLORS.get(int(l), ('#ffffff', f'Class {l}'))
                    x1, y1, x2, y2 = b
                    rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=2, edgecolor=color, facecolor='none')
                    ax.add_patch(rect)
                    ax.text(x1, max(0, y1 - 5), f'{cname}: {s:.2f}', color='white',
                            bbox=dict(boxstyle='square,pad=0.2', facecolor=color, alpha=0.85), fontsize=9)

        plt.tight_layout()
        plt.show()
